In [1]:
import cv2
import numpy as np
from reachy_sdk import ReachySDK

# --- CONFIGURATION PARAMETERS ---
# Number of INSIDE corners on your chessboard (Columns, Rows)
CHESSBOARD_SIZE = (9, 6)
# Size of a single square edge in millimeters
SQUARE_SIZE_MM = 25.0

# Prepare 3D object points based on real-world chessboard measurements
objp = np.zeros((CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE_MM

# Arrays to store vectors from all captured pairs
object_points = []  # 3d points in real world space
left_image_points = []   # 2d points in left image plane
right_image_points = []  # 2d points in right image plane

print("Connecting to Reachy 1.2...")
# Initialize Reachy SDK connection (Default IP is localhost)
robot = ReachySDK(host='10.22.129.133')

# Reachy 1.2 provides side-by-side or separate left/right video streams via SDK
left_cam = robot.left_camera
right_cam = robot.right_camera

print("\n--- INSTRUCTIONS ---")
print("1. Press [SPACE] to capture a synchronized frame pair.")
print("2. Move the chessboard to different angles and distances.")
print("3. Capture 20-30 good pairs for accurate calibration.")
print("4. Press [ESC] when done to run calibration calculation.")

capture_count = 0

while True:
    # Fetch synchronized frames from Reachy's head cameras
    frame_left = left_cam.read()
    frame_right = right_cam.read()
    
    if frame_left is None or frame_right is None:
        print("Error: Could not retrieve images from Reachy's cameras.")
        break

    # Clone frames to draw calibration feedback without changing the raw image array
    display_left = frame_left.copy()
    display_right = frame_right.copy()

    # Convert frames to grayscale for corner detection processing
    gray_left = cv2.cvtColor(frame_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(frame_right, cv2.COLOR_BGR2GRAY)

    # Search for chessboard corners in both video feeds
    ret_left, corners_left = cv2.findChessboardCorners(gray_left, CHESSBOARD_SIZE, None)
    ret_right, corners_right = cv2.findChessboardCorners(gray_right, CHESSBOARD_SIZE, None)

    # Visualize detected points live if the pattern is found
    if ret_left:
        cv2.drawChessboardCorners(display_left, CHESSBOARD_SIZE, corners_left, ret_left)
    if ret_right:
        cv2.drawChessboardCorners(display_right, CHESSBOARD_SIZE, corners_right, ret_right)

    # Combine displays side by side for a cleaner workspace view
    combined_view = np.hstack((display_left, display_right))
    cv2.putText(combined_view, f"Captured Pairs: {capture_count}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow("Reachy 1.2 Stereo Calibration (Left | Right)", combined_view)

    key = cv2.waitKey(1) & 0xFF
    
    # Save a synchronized snapshot frame pair if Spacebar is pressed
    if key == ord(' '):
        if ret_left and ret_right:
            # Refine pixel locations to sub-pixel accuracy levels
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
            corners_left_refined = cv2.cornerSubPix(gray_left, corners_left, (11, 11), (-1, -1), criteria)
            corners_right_refined = cv2.cornerSubPix(gray_right, corners_right, (11, 11), (-1, -1), criteria)

            object_points.append(objp)
            left_image_points.append(corners_left_refined)
            right_image_points.append(corners_right_refined)
            
            capture_count += 1
            print(f"Successfully captured pair #{capture_count}!")
        else:
            print("Warning: Chessboard not fully visible in BOTH camera views. Snapshot skipped.")

    # Break loop and execute calibration math if Escape key is hit
    elif key == 27:
        break

cv2.destroyAllWindows()

# Ensure we have gathered enough sample frames
if capture_count < 10:
    print(f"Error: Only {capture_count} pairs captured. You need at least 10 pairs to run calibration.")
else:
    print("\nProcessing calibration data... Please hold on.")
    image_shape = gray_left.shape[::-1] # (width, height)

    # Step 1: Initialize individual camera intrinsic approximations
    ret_l, K_l, D_l, _, _ = cv2.calibrateCamera(object_points, left_image_points, image_shape, None, None)
    ret_r, K_r, D_r, _, _ = cv2.calibrateCamera(object_points, right_image_points, image_shape, None, None)

    # Step 2: Perform comprehensive stereo calibration math
    # Computes spatial geometric link relationships between left and right sensor matrices
    flag = cv2.CALIB_FIX_INTRINSIC
    criteria_stereo = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-5)
    
    rmse, K_l, D_l, K_r, D_r, R, T, E, F = cv2.stereoCalibrate(
        object_points, left_image_points, right_image_points,
        K_l, D_l, K_r, D_r, image_shape,
        criteria=criteria_stereo, flags=flag
    )

    print("\n=== CALIBRATION RESULTS ===")
    print(f"Stereo Reprojection RMSE: {rmse:.4f} pixels (Aim for < 0.5)")
    print("\nLeft Camera Intrinsic Matrix (K_l):\n", K_l)
    print("\nRight Camera Intrinsic Matrix (K_r):\n", K_r)
    print("\nRotation Matrix (R) between eyes:\n", R)
    print("\nTranslation Vector (T) in mm:\n", T)

    # Save output parameters to disk for application runtime retrieval
    output_file = "reachy_stereo_params.npz"
    np.savez(output_file, K_l=K_l, D_l=D_l, K_r=K_r, D_r=D_r, R=R, T=T, E=E, F=F)
    print(f"\nSuccess! Stereo parameters securely saved to '{output_file}'.")


Connecting to Reachy 1.2...


/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:10.22.129.133:50055: Failed to connect to remote host: Connection refused"
	debug_error_string = "UNKNOWN:Error received from peer  {created_time:"2026-08-21T21:02:07.562180697+02:00", grpc_status:14, grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv4:10.22.129.133:50055: Failed to connect to remote host: Connection refused"}"
>